# OVA_Breast — Klasterovanje
### Seminarski rad iz predmeta Istraživanje podataka 2

**Skup podataka:** OVA_Breast (OpenML ID 1128, deo OVA/Ovarian kolekcije mikroarray
podataka genske ekspresije)
**Zadatak:** Klasterovanje

Notebook prati plan rada iz `PLAN.md` kroz 4 radna dana. Markdown ćelije obeležene sa
"🔵 GIT COMMIT" označavaju tačke gde se u stvarnom radu pravi git commit — dodate su
kao dokumentacija radnog toka, a ne kao izvršni kod. Ceo postupak je reproducibilan
pokretanjem "Run All" od početka do kraja (`random_state=42` svuda gde je primenljivo).


In [ ]:
import numpy as np
import pandas as pd
from scipy.io import arff
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import time

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

os.makedirs('../data', exist_ok=True)
os.makedirs('../output', exist_ok=True)
os.makedirs('../visualizations', exist_ok=True)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100


---
### 🔵 GIT COMMIT — Dan 1
```bash
git add .gitignore Readme.md requirements.txt
git commit -m "init: struktura projekta i gitignore"
git push
```
---

# DAN 1 — Setup i preprocesiranje

## 1. Uvod u podatke

Skup **OVA_Breast** sadrži mikroarray merenja genske ekspresije, gde svaki red predstavlja
jedan uzorak tkiva, a svaka kolona (osim `ID_REF` i `Tissue`) predstavlja nivo ekspresije
jednog gena (probeset-a). Ciljni atribut `Tissue` je binaran (`Breast` / `Other`) i
**koristi se isključivo za naknadnu (eksternu) evaluaciju klastera**, nikada kao ulaz u
sam algoritam klasterovanja — pošto je zadatak nenadgledano učenje.

Ključna karakteristika ovog skupa je **visoka dimenzionalnost u odnosu na broj uzoraka**
(tzv. HDLSS — high-dimension, low-sample-size problem): ~10935 atributa na svega ~1545
uzoraka. Ovo je centralni metodološki izazov analize i motiviše obaveznu redukciju
dimenzionalnosti pre klasterovanja.

## 2. Učitavanje podataka

Originalni `.arff` fajl (preuzet sa OpenML-a, https://openml.org/search?type=data&id=1128)
je ~108MB, preko GitHub limita od 100MB po fajlu — zato se ne verzioniše (videti
`.gitignore`) i mora se lokalno postaviti u `data/OVA_Breast.arff` pre pokretanja.


In [ ]:
ARFF_PATH = '../data/OVA_Breast.arff'

data, meta = arff.loadarff(ARFF_PATH)
df_raw = pd.DataFrame(data)

# Kategoricke/string kolone se u scipy.io.arff ucitavaju kao bytes objekti - dekodiramo ih
df_raw['Tissue'] = df_raw['Tissue'].str.decode('utf-8')

print(f"Oblik podataka: {df_raw.shape}")
print(f"Kolone (prvih 5): {df_raw.columns[:5].tolist()}")
print(f"Kolone (poslednje 3): {df_raw.columns[-3:].tolist()}")


## 3. Osnovna statistika podataka

In [ ]:
print("=== Osnovne informacije ===")
print(f"Broj instanci: {df_raw.shape[0]}")
print(f"Broj atributa (ukupno, sa ID_REF i Tissue): {df_raw.shape[1]}")
print(f"Broj gen-ekspresija atributa: {df_raw.shape[1] - 2}")
print()
print("=== Raspodela ciljnog atributa Tissue ===")
print(df_raw['Tissue'].value_counts())
print()
print(df_raw['Tissue'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = df_raw['Tissue'].value_counts()
colors = ['#D85A30', '#378ADD']
ax.bar(counts.index, counts.values, color=colors)
ax.set_ylabel('Broj uzoraka')
ax.set_title('Raspodela klasa - Tissue (samo informativno,\nne koristi se u klasterovanju)')
for i, v in enumerate(counts.values):
    ax.text(i, v + 15, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/class_distribution.png', dpi=150)
plt.show()

print(f"\nNapomena: klase su neuravnotezene ({counts.iloc[0]} vs {counts.iloc[1]}, "
      f"odnos ~{counts.iloc[0]/counts.iloc[1]:.1f}:1). Ovo je bitno pri tumacenju "
      f"eksterne evaluacije klastera - klasteri se ne moraju poklapati 1:1 sa ovom podelom.")


In [ ]:
missing = df_raw.isnull().sum().sum()
print(f"Ukupan broj nedostajucih vrednosti: {missing}")

dup_ids = df_raw['ID_REF'].duplicated().sum()
print(f"Broj dupliranih ID_REF vrednosti: {dup_ids}")

dup_rows = df_raw.drop(columns=['ID_REF']).duplicated().sum()
print(f"Broj potpuno dupliranih redova (bez ID_REF): {dup_rows}")

sample_cols = df_raw.drop(columns=['ID_REF', 'Tissue']).sample(5, axis=1, random_state=RANDOM_STATE).columns
print("\nRaspon vrednosti za 5 nasumicno izabranih gena (ilustracija heterogenosti):")
print(df_raw[sample_cols].describe().loc[['min', 'mean', 'max']].T)


**Zapažanje:** Nema nedostajućih vrednosti niti dupliranih redova — ne zahteva se
imputacija ili uklanjanje duplikata. Međutim, vrednosti genske ekspresije variraju u
ekstremno velikim rasponima između različitih gena, što **obavezno zahteva
standardizaciju** pre bilo koje metode zasnovane na rastojanju.

Skup nema tekstualnih kolona (npr. `tags`), pa korak obrade teksta iz opšteg uputstva
predmeta ovde nije primenjiv — svi atributi su već numerički (nivoi ekspresije).
Takođe, `Tissue` je jedini ciljni/kategorički atribut, pa nema potrebe za više
varijanti ciljnog atributa.


---
### 🔵 GIT COMMIT — Dan 1
```bash
git add notebooks/ visualizations/class_distribution.png
git commit -m "feat: ucitavanje arff i osnovna statistika"
git push
```
---

## 4. Preprocesiranje

Koraci:
1. Uklanjanje `ID_REF` (identifikator, nije informativan atribut)
2. Odvajanje `Tissue` labele (čuva se odvojeno, isključivo za evaluaciju)
3. Standardizacija preostalih atributa (`StandardScaler`)


In [ ]:
y_tissue = df_raw['Tissue'].copy()
X_raw = df_raw.drop(columns=['ID_REF', 'Tissue'])

print(f"Oblik matrice atributa (X): {X_raw.shape}")
print(f"Oblik labele (y, samo za evaluaciju): {y_tissue.shape}")


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_scaled = pd.DataFrame(X_scaled, columns=X_raw.columns, index=X_raw.index)

print("Standardizacija zavrsena.")
print(f"Prosek po koloni posle skaliranja (treba biti ~0): {X_scaled.values.mean():.6f}")
print(f"Std po koloni posle skaliranja (treba biti ~1): {X_scaled.values.std():.6f}")


In [ ]:
# NAPOMENA: pun standardizovani skup (10935 atributa) je i kao .pkl ~135MB, sto je
# preko GitHub limita od 100MB po fajlu. Zato se ovaj fajl NE verzionise (dodat je u
# .gitignore) - lokalno se regenerise pokretanjem ove celije. U git idu samo manji,
# redukovani skupovi (PCA-50, TopVar-200) koji se stvarno koriste za klasterovanje.
df_preprocessed = X_scaled.copy()
df_preprocessed['Tissue'] = y_tissue.values

df_preprocessed.to_pickle('../data/data_preprocessed_full.pkl')
size_mb = os.path.getsize('../data/data_preprocessed_full.pkl') / (1024**2)
print(f"Sacuvano: data/data_preprocessed_full.pkl, oblik: {df_preprocessed.shape}, velicina: {size_mb:.1f} MB")


## 5. Redukcija dimenzionalnosti

Zbog HDLSS prirode podataka (10935 atributa naspram 1545 uzoraka), klasterovanje na
punom skupu atributa je podložno "prokletstvu dimenzionalnosti". Zato kreiramo i
redukovane reprezentacije, koje ćemo koristiti paralelno sa punim skupom u svim
narednim koracima klasterovanja — ovo je i eksplicitan zahtev iz uputstva predmeta
(poređenje modela na punom i redukovanom skupu atributa).


In [ ]:
from sklearn.decomposition import PCA

pca_full = PCA(random_state=RANDOM_STATE).fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

for target in [0.5, 0.7, 0.9, 0.95]:
    n_comp = np.argmax(cumvar >= target) + 1
    print(f"{target*100:.0f}% objasnjene varijanse -> potrebno {n_comp} komponenti")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(cumvar) + 1), cumvar, color='#0F6E56')
ax.axhline(0.9, color='#993C1D', linestyle='--', linewidth=1, label='90% varijanse')
ax.set_xlabel('Broj glavnih komponenti')
ax.set_ylabel('Kumulativna objasnjena varijansa')
ax.set_title('PCA - kumulativna objasnjena varijansa')
ax.legend()
plt.tight_layout()
plt.savefig('../visualizations/pca_explained_variance.png', dpi=150)
plt.show()


In [ ]:
# Skup 1: PCA - 50 komponenti (standardan izbor, uporediv sa referentnim radom)
N_COMPONENTS = 50
pca_50 = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
X_pca50 = pca_50.fit_transform(X_scaled)

print(f"PCA (50 komponenti) oblik: {X_pca50.shape}")
print(f"Objasnjena varijansa: {pca_50.explained_variance_ratio_.sum()*100:.2f}%")

df_pca50 = pd.DataFrame(X_pca50, columns=[f'PC{i+1}' for i in range(N_COMPONENTS)])
df_pca50['Tissue'] = y_tissue.values
df_pca50.to_csv('../data/data_preprocessed_pca50.csv', index=False)
print("Sacuvano: data/data_preprocessed_pca50.csv")


In [ ]:
# Skup 2: Top-K najvarijabilnijih gena (uobicajen pristup kod gene-expression podataka -
# geni sa najvecom varijansom nose najvise informacija o razlikama izmedju uzoraka)
TOP_K = 200
variances = X_scaled.var(axis=0)
top_genes = variances.sort_values(ascending=False).head(TOP_K).index

X_topvar = X_scaled[top_genes]
print(f"Skup top-{TOP_K} najvarijabilnijih gena, oblik: {X_topvar.shape}")

df_topvar = X_topvar.copy()
df_topvar['Tissue'] = y_tissue.values
df_topvar.to_csv('../data/data_preprocessed_topvar200.csv', index=False)
print(f"Sacuvano: data/data_preprocessed_topvar200.csv")


**Rezime kreiranih skupova atributa** (koriste se paralelno u klasterovanju radi poređenja):

| Naziv | Opis | Broj atributa | Fajl |
|---|---|---|---|
| Full | Svi standardizovani atributi | 10935 | `data_preprocessed_full.pkl` (lokalno, nije u git-u) |
| PCA-50 | PCA redukcija na 50 komponenti | 50 | `data_preprocessed_pca50.csv` |
| TopVar-200 | Top 200 gena po varijansi | 200 | `data_preprocessed_topvar200.csv` |


---
### 🔵 GIT COMMIT — Dan 1
```bash
git add data/data_preprocessed_pca50.csv data/data_preprocessed_topvar200.csv output/scaler.pkl output/pca_50.pkl output/top_var_genes.pkl visualizations/pca_explained_variance.png
git commit -m "data: standardizacija i PCA/TopVar redukcija dimenzionalnosti"
git push
```
---

## 6. Vizuelizacija podataka (2D i 3D)

Pre klasterovanja, vizuelno ispitujemo strukturu podataka projektovanjem u 2D i 3D
prostor pomoću PCA. Tačke su obojene prema `Tissue` labeli isključivo radi vizuelne
orijentacije — ovo NIJE deo procesa klasterovanja.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D

pca_3 = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca3 = pca_3.fit_transform(X_scaled)

colors_map = {'Breast': '#D85A30', 'Other': '#378ADD'}

fig = plt.figure(figsize=(13, 5))

ax1 = fig.add_subplot(1, 2, 1)
for label, color in colors_map.items():
    mask = (y_tissue == label).values
    ax1.scatter(X_pca3[mask, 0], X_pca3[mask, 1], c=color, label=label, alpha=0.6, s=15)
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')
ax1.set_title('2D PCA projekcija (obojeno po Tissue)')
ax1.legend()

ax2 = fig.add_subplot(1, 2, 2, projection='3d')
for label, color in colors_map.items():
    mask = (y_tissue == label).values
    ax2.scatter(X_pca3[mask, 0], X_pca3[mask, 1], X_pca3[mask, 2], c=color, label=label, alpha=0.6, s=15)
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')
ax2.set_zlabel('PC3')
ax2.set_title('3D PCA projekcija (obojeno po Tissue)')
ax2.legend()

plt.tight_layout()
plt.savefig('../visualizations/pca_2d_3d.png', dpi=150)
plt.show()


**Zapažanje:** Nema jasne linearne separabilnosti između klasa `Breast` i `Other` u
prve dve/tri glavne komponente — tačke se preklapaju u velikoj meri. Ovo je važno za
tumačenje kasnijih rezultata klasterovanja: ne treba očekivati da će nenadgledani
klasteri savršeno odgovarati ovoj podeli, pošto ni sama PCA projekcija ne pokazuje
prirodnu binarnu strukturu.


In [ ]:
with open('../output/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('../output/pca_50.pkl', 'wb') as f:
    pickle.dump(pca_50, f)
with open('../output/top_var_genes.pkl', 'wb') as f:
    pickle.dump(list(top_genes), f)

print("Sacuvani pomocni objekti: scaler.pkl, pca_50.pkl, top_var_genes.pkl")
print("\nDAN 1 zavrsen. Spremno za Dan 2 - klasterovanje.")


---
### 🔵 GIT COMMIT — Dan 1
```bash
git add visualizations/pca_2d_3d.png
git commit -m "viz: 2D/3D PCA projekcije obojene po Tissue labeli"
git push
```
---